# Steam Data Collection

Collects the reviews, discussion posts and comments of the Steam pages listed in `steam_links.txt`. Steam has no search over reviews and comments, so the pages are chosen by hand, as for Instagram and TikTok.

No Steam account or login is needed: the data is read from public Steam Community pages, the comment endpoint those pages use, and Steam's public review API.

Supported links:
1. A profile's review list (`https://steamcommunity.com/id/<name>/recommended/`): every review on all its pages, each with its comments.
2. A single review (`.../recommended/<app id>/`): the review and its comments.
3. A profile or its comments page (`.../allcomments`): all profile comments.
4. A group (`https://steamcommunity.com/groups/<name>`): all group comments.
5. A discussion forum (`https://steamcommunity.com/app/<app id>/discussions/`): the newest topics, each with its replies.
6. A discussion topic (`.../discussions/<forum>/<topic>/`): the first post and all replies.
7. A store page (`https://store.steampowered.com/app/<app id>/`): store reviews of the game from the past year, as many as Steam's review API returns. For some games, such as Dota 2, the API returns only a few dozen.

Page and language parameters in a link (`?p=2`, `?ctp=3`, `?l=malay`) are ignored, since all pages are collected. Steam has no Persian review language, and most texts on large forums and store pages are in other languages, so by default only Persian texts are kept. A text counts as Persian when it is in Arabic script but has none of the letters only Arabic uses (ة and the Arabic forms of ي and ك), so Persian typed with an Arabic keyboard layout and short Arabic texts can be misjudged. Private profiles are reported as failed links.

In [ ]:
import itertools
import json
import re
import time
from urllib.parse import urlsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configuration

In [ ]:
LINKS_FILE = "steam_links.txt"
OUTPUT_CSV = "SteamComments.csv"
KEEP_ONLY_PERSIAN = True  # large forums and store reviews are mostly in other languages
MAX_COMMENTS_PER_THREAD = 2000  # comments per review, profile, group or discussion topic
MAX_REVIEWS_PER_PROFILE = 200  # reviews read from a profile's review list
MAX_TOPICS_PER_FORUM = 30  # newest topics read from a discussion forum
MAX_REVIEWS_PER_GAME = 2000  # store reviews scanned per game; only a few of them are Persian
REVIEW_LANGUAGES = ['english', 'arabic']  # Steam has no Persian review language; Persian store reviews are mostly labeled English
DELAY_BETWEEN_REQUESTS = 3  # seconds; Steam answers 429 Too Many Requests after a few fast requests
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

## Read links

In [ ]:
def read_links(path):
    # One link per line; blank lines and lines starting with # are ignored
    with open(path, 'r', encoding='utf-8') as links_file:
        lines = [line.strip() for line in links_file]
    return [line for line in lines if line and not line.startswith('#')]

def normalize_link(link):
    # Drops page and language parameters (?p=2, ?ctp=3, ?l=malay) because every page is collected anyway
    parts = urlsplit(link)
    return 'https://' + parts.netloc + parts.path.rstrip('/') + '/'

links = list(dict.fromkeys(normalize_link(link) for link in read_links(LINKS_FILE)))  # duplicates removed, file order kept
len(links)

## Request Steam pages

In [ ]:
def make_session():
    # Retries rate limit and server errors, waiting 10 s, 20 s, 40 s and so on up to 5 minutes between attempts,
    # because Steam can keep answering 429 for several minutes
    retry = Retry(total=8, backoff_factor=10, backoff_max=300, status_forcelist=[429, 500, 502, 503, 504])
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers.update({'User-Agent': USER_AGENT})
    return session

def get(session, url, params=None):
    time.sleep(DELAY_BETWEEN_REQUESTS)
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response

def get_page(session, url, params=None):
    # Pages are requested in English so that dates have one format.
    # Steam shows errors such as a missing profile as a normal page with a message.
    html = get(session, url, {'l': 'english', **(params or {})}).text
    soup = BeautifulSoup(html, 'html.parser')
    error = soup.select_one('#message h3')
    if error is not None:
        raise ValueError('Steam error: ' + error.get_text(strip=True))
    if soup.select_one('.profile_private_info') is not None:
        raise ValueError('The profile is private')
    return html, soup

## Get reviews, discussion posts and comments

In [ ]:
THREAD_PATTERN = re.compile(r'InitializeCommentThread\(\s*"\w+",\s*"[^"]*",\s*')
THREAD_URL_PATTERN = re.compile(r"\s*,\s*'([^']+)'")
COMMENTS_PER_REQUEST = 50  # the most comments Steam returns per request

def find_comment_threads(html):
    # Each comment thread of a page is set up by a script call with its settings as JSON, e.g.
    # InitializeCommentThread( "Profile", "Profile_7656...", {"feature":"-1","feature2":-1,"owner":"7656...",...}, 'https://steamcommunity.com/comment/Profile/', 40 )
    threads = []
    for match in THREAD_PATTERN.finditer(html):
        settings, end = json.JSONDecoder().raw_decode(html, match.end())
        url_match = THREAD_URL_PATTERN.match(html, end)
        if url_match is None:
            raise ValueError('Unknown comment thread format: ' + html[match.start():end + 100])
        render_url = url_match.group(1) + 'render/' + settings['owner'] + '/' + settings['feature'] + '/'
        params = {'feature2': settings['feature2'], 'extended_data': settings.get('extended_data', '')}
        threads.append((render_url, params))
    return threads

def iter_thread_comments(session, render_url, params):
    start = 0
    while True:
        page = get(session, render_url, {**params, 'start': start, 'count': COMMENTS_PER_REQUEST}).json()
        if not page.get('success'):
            raise ValueError('Steam did not return the comments of ' + render_url + ': ' + str(page.get('error')))
        comments = BeautifulSoup(page['comments_html'], 'html.parser').select('.commentthread_comment')
        yield from comments
        start += len(comments)
        if not comments or start >= page['total_count']:
            return

In [ ]:
REVIEW_ID_PATTERN = re.compile(r"UserReviewVoteUp\(\s*\d+,\s*'[^']*',\s*'(\d+)'")
REVIEW_DATE_PATTERN = re.compile(r'Posted: (\d{1,2} \w+)(?:, (\d{4}))? @ (\d{1,2}:\d{2}[ap]m)')

def element_text(element):
    # Line breaks are kept as new lines; quotes of earlier posts are removed so their text is not collected twice
    for quote in element.select('blockquote'):
        quote.decompose()
    for line_break in element.select('br'):
        line_break.replace_with('\n')
    return element.get_text().strip()

def timestamp_of(element):
    return pd.to_datetime(int(element.select_one('[data-timestamp]')['data-timestamp']), unit='s')

def parse_review_date(text):
    # e.g. 'Posted: 11 Dec, 2022 @ 12:26pm'; Steam leaves the year out for reviews of the current year
    match = REVIEW_DATE_PATTERN.search(text)
    if match is None:
        raise ValueError('Unknown review date format: ' + text)
    day_month, year, clock = match.groups()
    return pd.to_datetime(f'{day_month} {year or pd.Timestamp.now().year} {clock}')

def text_row(link, page_url, source, text_id, username, created_at, text):
    return {'link': link, 'page_url': page_url, 'source': source, 'id': text_id, 'username': username,
            'created_at': created_at, 'text': text}

def comment_row(link, page_url, comment):
    return text_row(link, page_url, 'comment', comment['id'].removeprefix('comment_'),
                    comment.select_one('.commentthread_author_link').get_text(strip=True), timestamp_of(comment),
                    element_text(comment.select_one('.commentthread_comment_text')))

def review_row(link, page_url, html, soup):
    review = soup.select_one('#ReviewText')
    if review is None:
        return None
    review_id = REVIEW_ID_PATTERN.search(html)
    if review_id is None:
        raise ValueError('No review id in ' + page_url)
    return text_row(link, page_url, 'review', review_id.group(1),
                    soup.select_one('.profile_small_header_name').get_text(strip=True),
                    parse_review_date(soup.select_one('.recommendation_date').get_text(' ', strip=True)),
                    element_text(review))

def topic_row(link, page_url, soup):
    topic = soup.select_one('.forum_op')
    if topic is None:
        return None
    text = element_text(topic.select_one('.topic')) + '\n' + element_text(topic.select_one('.content'))
    return text_row(link, page_url, 'topic', topic['id'].removeprefix('forum_op_'),
                    topic.select_one('.forum_op_author').get_text(strip=True), timestamp_of(topic), text)

def get_page_texts(session, link, page_url):
    # The review or discussion topic of the page, if it has one, and the comments of all its comment threads
    html, soup = get_page(session, page_url)
    rows = [row for row in (review_row(link, page_url, html, soup), topic_row(link, page_url, soup)) if row is not None]
    for render_url, params in find_comment_threads(html):
        comments = itertools.islice(iter_thread_comments(session, render_url, params), MAX_COMMENTS_PER_THREAD)
        rows.extend(comment_row(link, page_url, comment) for comment in comments)
    return rows

In [ ]:
GAME_REVIEWS_URL = 'https://api.steampowered.com/IUserReviewsService/GetAppReviews/v1/'
STORE_APP_PATTERN = re.compile(r'store\.steampowered\.com/app/(\d+)/')
FORUM_PATTERN = re.compile(r'steamcommunity\.com/(?:app|groups)/[^/]+/discussions/(?:\d+/)?$')
REVIEW_LIST_PATTERN = re.compile(r'steamcommunity\.com/(?:id|profiles)/[^/]+/recommended/$')

def iter_list_links(session, list_url, page_param, link_selector):
    # Lists are split into pages (?p=2 or ?fp=2); a page past the last one is empty
    for page in itertools.count(1):
        _, soup = get_page(session, list_url, {page_param: page})
        item_links = [anchor['href'] for anchor in soup.select(link_selector)]
        if not item_links:
            return
        yield from item_links

def iter_game_reviews(session, app_id):
    # Reviews of the past year (without day_range the API only covers the last 30 to 90 days), 100 per request;
    # each response has the cursor of the next page
    params = {'appid': app_id, 'num_per_page': 100, 'cursor': '*', 'filter': 'recent', 'day_range': 365}
    params.update({f'languages[{i}]': language for i, language in enumerate(REVIEW_LANGUAGES)})
    while True:
        page = get(session, GAME_REVIEWS_URL, params).json()['response']
        reviews = page.get('reviews', [])
        yield from reviews
        if not reviews or page.get('cursor', params['cursor']) == params['cursor']:
            return
        params['cursor'] = page['cursor']

def game_review_row(link, app_id, review):
    steam_id = review['author']['steamid']
    page_url = f'https://steamcommunity.com/profiles/{steam_id}/recommended/{app_id}/'
    return text_row(link, page_url, 'review', review['recommendationid'], steam_id,
                    pd.to_datetime(review['timestamp_created'], unit='s'), review['review'])

def get_link_texts(session, link):
    store_app = STORE_APP_PATTERN.search(link)
    if store_app:
        app_id = store_app.group(1)
        reviews = itertools.islice(iter_game_reviews(session, app_id), MAX_REVIEWS_PER_GAME)
        return [game_review_row(link, app_id, review) for review in reviews]
    if FORUM_PATTERN.search(link):
        page_urls = itertools.islice(iter_list_links(session, link, 'fp', 'a.forum_topic_overlay'), MAX_TOPICS_PER_FORUM)
    elif REVIEW_LIST_PATTERN.search(link):
        page_urls = itertools.islice(iter_list_links(session, link, 'p', '.review_box .title a'), MAX_REVIEWS_PER_PROFILE)
    elif 'steamcommunity.com/' in link:
        page_urls = [link]
    else:
        raise ValueError('Not a supported Steam link: ' + link)
    rows = []
    for page_url in page_urls:
        rows.extend(get_page_texts(session, link, page_url))
    return rows

In [ ]:
session = make_session()
all_texts = []
failed_links = []
for i, link in enumerate(links):
    print(i, '--------------- getting texts of', link, '------------------')
    try:
        link_texts = get_link_texts(session, link)
        all_texts.extend(link_texts)
        print('got', len(link_texts), 'texts')
    except (requests.RequestException, ValueError) as err:
        print('failed:', err)
        failed_links.append(link)

print('total texts:', len(all_texts), '| failed links:', len(failed_links))

In [ ]:
ARABIC_SCRIPT_LETTERS = re.compile('[ء-يٮ-ۓ]')  # includes the Persian letters
ARABIC_ONLY_LETTERS = re.compile('[ةيك]')  # Arabic teh marbuta, yeh and kaf; Persian writes ی and ک

def is_persian(text):
    # A simple letter check: Arabic script, but none of the letters that only Arabic uses
    return bool(ARABIC_SCRIPT_LETTERS.search(text)) and not ARABIC_ONLY_LETTERS.search(text)

def clean_text(text):
    # same cleaning as the X, Instagram and TikTok data: remove #, commas, new lines and @mentions
    text = text.replace('#', '').replace(',', ' ').replace('\r', ' ').replace('\n', ' ')
    return re.sub(r'@[^\s]+', '', text).strip()

if not all_texts:
    raise ValueError('No texts were collected; check the links in failed_links')

steam_df = pd.DataFrame(all_texts).drop_duplicates(subset=['source', 'id'])  # e.g. a review linked on its own and in a review list
steam_df['text'] = steam_df['text'].apply(clean_text)
steam_df = steam_df[steam_df['text'] != '']
if KEEP_ONLY_PERSIAN:
    steam_df = steam_df[steam_df['text'].apply(is_persian)]

steam_df.to_csv(OUTPUT_CSV, mode='w', header=True, index=True, encoding='utf-8')

print('kept texts:', len(steam_df))
steam_df.head(900)